# Manejo básico de datos con Spark

En este ejercicio vamos a repasar algunas operaciones de manejo de datos
con Spark para comprobar qué tal te desenvuelves. De nuevo, seguiremos
los mismos pasos que ya completamos tanto con R como con Pandas y
Python. La idea, más que proponerte un ejercicio complicado, es que esto
te permita comparar cómo se realizan distintas tareas en Spark, frente a
las formas de hacerlo en los lenguajes que ya vimos.

Tendrás que iniciar sesión en la máquina virtual y arrancar la consola
de PySpark.

Una vez arrancada la sesión en la consola de PySpark, vamos a leer de
fichero los datos de consumo horario de electricidad de un cliente. Se
trata de un fichero en formato CSV, con un encabezado en la primera
línea y con el carácter punto y coma para separar los campos.

#### P1. Completa el comando para leer los datos del fichero indicado en la variable consumo\_file

In [1]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

consumo_file = r'file:///home/bigdata/Curso/U13_Spark/datos/consumo_horario_cliente.csv'

consumo = spark.read.csv(path = consumo_file, sep = ";", header =True, inferSchema = True)

NameError: name 'spark' is not defined

Comprobemos que los datos se han cargado bien

In [ ]:
consumo.show(5)

#### P2. Imprime el esquema del DataFrame consumo

In [ ]:
consumo.printSchema()

#### P2.2 Selecciona una columna del DataFrame

In [ ]:
consumo.select("datetime")

Trabajando con fechas y horas en Spark
--------------------------------------

Si no ha habido ningún problema, la columna datetime ha de tener el tipo
timestamp. Al cargar datos, **Spark puede inferir el tipo de las columnas
analizando su contenido. Esto puede requerir que usemos la opción
inferSchema=True al leer los datos.** Si no la usaste, puedes volver a
cargar el contenido del fichero incluyéndola ahora.

En este caso, Spark ha sabido detectar que la columna datetime contiene
información con formato de fecha y hora, por lo que transforma los
valores directamente a un tipo timestamp.

Que Spark convierta directamente la columna al tipo timestamp en lugar
de dejarla como cadena de texto nos permite poder realizar operaciones
de aritmética de fecha y hora, o extraer partes de la fecha y hora de
forma sencilla. Para ello, dentro de la amplia lista de funciones
incluidas en el módulo pyspark.sql.functions
(<http://spark.apache.org/docs/latest/api/python/pyspark.sql.html#module-pyspark.sql.functions>),
Spark nos ofrece varias funciones específicas para trabajar con objetos
timestamp

|  **Función**     |  **Descripción** |
|  -----------------| ----------------------------------------------------------------|
|  year(col)        | Extraer el año (número) de una fecha en la columna col |
|  month(col)       | Extraer el mes (número) de una fecha en la columna col |
|  dayofmonth(col)  | Extraer el día del mes (número) de una fecha en la columna col |
|  hour(col)        | Extraer la hora de un timestamp en la columna col |
|  minute(col)      | Extraer los minutos de un timestamp en la columna col |
|  to_date(col)     | Convertir la columna col a tipo fecha (año-mes-día) |

También disponemos de otra función, ***F.date_format()***, a la que le pasamos
la columna y una cadena de texto para especificar el formato que
queremos dar a la fecha o timestamp:

In [ ]:
consumo.select("datetime").withColumn('hhmm', F.date_format(col("datetime"), "HHmm")).show(5)

~~~
+--------------------+----+
|            datetime|hhmm|
+--------------------+----+
|2016-09-01 00:00:...|0000|
|2016-09-01 01:00:...|0100|
|2016-09-01 02:00:...|0200|
|2016-09-01 03:00:...|0300|
|2016-09-01 04:00:...|0400|
+--------------------+----+
only showing top 5 rows

~~~

In [ ]:
consumo.select("datetime").withColumn('dia_semana', F.date_format(col("datetime"), "E")).show(5)

~~~
+--------------------+----------+
|            datetime|dia_semana|
+--------------------+----------+
|2016-09-01 00:00:...|       jue|
|2016-09-01 01:00:...|       jue|
|2016-09-01 02:00:...|       jue|
|2016-09-01 03:00:...|       jue|
|2016-09-01 04:00:...|       jue|
+--------------------+----------+
only showing top 5 rows
~~~

**Esta funcion necesita 2 argumentos: la *columna* donde estan los datos y el *formato* que tendran los datos**

El formato puede ser uno de la lista que aparece a continuacion



|  **Letter** |  **Date or Time Component**    | **Presentation** | **Examples** |
|  ------------| -------------------------------|-----------------|-------|
|G|Era designator|[Text](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#text)|AD|
|y|Year |[Year](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#year)|1996; 96|
|Y|Week year|[Year](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#year)|2009; 09|
|M|Month in year|[Month](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#month)|July; Jul; 07|
|w|Week in year|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|27|
|W|Week in month|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|2|
|D|Day in year|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|189|
|d|Day in month|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|10|
|F|Day of week in month|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|2|
|E|Day name in week|[Text](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#text)|Tuesday; Tue|
|u|Day number of week (1 = Monday, ..., 7 = Sunday)|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|1|
|a|Am/pm marker|[Text](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#text)|PM|
|H|Hour in day (0-23)|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|0|
|k|Hour in day (1-24)|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|24|
|K|Hour in am/pm (0-11)|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|0|
|h|Hour in am/pm (1-12)|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|12|
|m|Minute in hour|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|30|
|s|Second in minute|[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|55|
|S|Millisecond |[Number](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#number)|978|
|z|Time zone|[General time zone](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#timezone)|Pacific Standard Time; PST; GMT-08:00|
|Z|Time zone|[RFC 822 time zone](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#rfc822timezone)|-0800|
|X|Time zone|[ISO 8601 time zone](https://docs.oracle.com/javase/7/docs/api/java/text/SimpleDateFormat.html#iso8601timezone)|-08; -0800; -08:00|

#### P3. Ayudándote de estas funciones, modifica el DataFrame de consumos añadiendo las siguientes columnas:

-   fecha : extrayendo la fecha de la columna datetime

-   hora : extrayendo la hora de la columna datetime

-   dia\_semana : extrayendo el día de la semana de la columna datetime

-   mes: extrayendo el mes de la columna datetime

-   coste\_eur : con el coste en euros de la energía consumida,
    multiplicando el consumo por el precio

In [ ]:
# Completa el código
consumo_ext = consumo
.withColumn("fecha", F.to_date("datetime"))
.withColumn("hora", F.hour("datetime"))
.withColumn("dia", F.date_format(col("datetime"),"E"))
.withColumn("mes", F.month("datetime"))
.withColumn("coste_eur",( F.col("consumo_kwh")* F.col("precio_kwh")))

Al final deberías obtener algo así:

In [ ]:
consumo_ext.show(5)

~~~
+--------------------+-----------+-----------+----------+----+----------+---+--------------+
|            datetime|consumo_kwh| precio_kwh|     fecha|hora|dia_semana|mes|     coste_eur|
+--------------------+-----------+-----------+----------+----+----------+---+--------------+
|2016-09-01 00:00:...|      0.188|    0.09751|2016-09-01|   0|       jue|  9|       0.01833|
|2016-09-01 01:00:...|      0.124|    0.09775|2016-09-01|   1|       jue|  9|       0.01212|
|2016-09-01 02:00:...|      0.111|    0.09771|2016-09-01|   2|       jue|  9|       0.01084|
|2016-09-01 03:00:...|      0.129|    0.09927|2016-09-01|   3|       jue|  9|       0.01280|
|2016-09-01 04:00:...|      0.108|     0.1007|2016-09-01|   4|       jue|  9|       0.01088|
+--------------------+-----------+-----------+----------+----+----------+---+--------------+
only showing top 5 rows
~~~

#### P4. ¿Cuál es el consumo máximo que se ha registrado en los datos que tenemos?

In [ ]:
# escribe aquí el código
consumo.agg(F.max("precio_kwh").alias("maximo_:precio")).show()

> Para realizar funciones estadisticas (*media, max, min, moda,...*) **siempre sera necesario utilizar o un ***.groupBy() o un .agg()***

Recolectando resultados distribuidos de Spark
---------------------------------------------

Antes de continuar, ¿qué hacemos si lo que queremos es utilizar
directamente un valor calculado o extraído de un DataFrame como si fuera
un valor normal? Por ejemplo, si queremos almacenar el consumo máximo en
una variable normal, y no en un DataFrame o un RDD.

Existe un método en Spark que no hemos visto en la unidad: el método
***.collect()***. Este método fuerza a Spark a **recopilar todos los bloques o
particiones de un DataFrame o RDD que están distribuidos por los nodos
del cluster, y unificarlos en un array accesible** por el proceso ejecutor
(driver) como una variable de tipo lista.

Veamos un ejemplo. La expresión para obtener el consumo máximo devuelve
un DataFrame de una columna.

In [ ]:
consumo_ext.agg(F.max(consumo_ext.consumo_kwh))

`DataFrame[max(consumo_kwh): double]`

**Si hacemos un ***.collect()***, obtenemos una lista de objetos Row, que representan filas del DataFrame original.**

> Cada objeto Row contiene los valores correspondientes para cada columna.

In [ ]:
max_consumo = consumo_ext.agg(F.max(consumo_ext.consumo_kwh)).collect()

max_cons

`[Row(max(consumo_kwh)=3.608)]`

In [ ]:
type(max_consumo)

`list`

Como el resultado de collect() es una lista, **podemos acceder a sus
elementos usando índices, como en cualquier otra lista en Python.**

In [ ]:
max_consumo[0]

`Row(max(consumo_kwh)=3.608)`

In [ ]:
type(max_consumo[0])

`pyspark.sql.types.Row`

**Los valores de cada columna de un objeto Row pueden accederse también
usando un índice (como en una lista), o usando el nombre de la columna**

In [ ]:
max_consumo[0][0]

`3.608`

In [ ]:
max_consumo[0]['max(consumo_kwh)']

`3.608`

#### P5. Extrae del DataFrame la fila correspondiente al día y hora en el que se ha registrado el mayor consumo. No uses directamente el valor númerico para filtrar. En su lugar, usa la expresión para calcular el máximo consumo.

In [ ]:
# escribe aquí el código
consumo_max = consumo_ext.where(F.col("consumo_kwh") == (consumo_ext.agg(F.max(consumo_ext.consumo_kwh))).collect()[0][0]).show()

> El codigo se ha creado asi:
>
> > ***consumo_ext*** es el dataframe creado con las columnas adicionales
> >
> > ***.where(*** selecciona **las filas de la columnas que cumplan la condicion que indiquemos**
> >
> > ***(F.col("consumo_kwh")*** es la columna que tiene que cumplir la condicion
> >
> > ***== (consumo_ext.agg(F.max(consumo_ext.consumo_kwh)*** es la condicion, que permite generar un agregado para encontrar valores maximos.
> >
> > ***.collect()[0][0])*** extrae el valor especifico maximo (que estara siempre en la posicion [0][0])

Trabajando con SQL
------------------

Ahora vamos a tratar de resolver algunos de los problemas previos
mediante la interfaz SQL en lugar de operar directamente sobre los
DataFrames.

#### P6. Resuelve la pregunta P3, pero esta vez registra el DataFrame consumo como una vista en Spark, y utiliza una consulta SQL para generar el mismo resultado que en P3. Comprueba que el resultado es efectivamente el mismo. Registra el resultado también como una vista.

Pista: en las consultas SQL sobre Spark podemos usar directamente todas
las funciones del módulo pyspark.sql.functions, no es necesario
importarlas ni acceder a través de un alias.

In [ ]:
# escribe aquí el código


#### P7. De forma análoga, extrae de la vista anterior la fila correspondiente al día y hora en el que se ha registrado el mayor consumo utilizando una consulta SQL. Compara el resultado con lo obtenido en la pregunta P5.

Pista: Spark SQL nos permite también usar subconsultas, igual que en la
mayoría de bases de datos. Puedes usar una subconsulta para obtener el
consumo máximo, y usarla para filtrar la vista en la consulta principal.

In [ ]:
# escribe aquí el código


Trabajando con varias tablas o DataFrames
-----------------------------------------

Ahora vamos a cargar los datos de precio de la electricidad en el
mercado diario mayorista (distinto del precio al que se cobra la
electricidad al consumidor).

Se trata de un fichero en formato CSV, con un encabezado en la primera
línea y con el carácter punto y coma para separar los campos.

#### P8. Completa el comando para leer los datos del fichero indicado en la variable pmd\_file

In [ ]:
pmd_file = r'file:///home/bigdata/Curso/U13\_Spark/datos/precio\_md.csv'

pmd = spark.read.csv(path = pmd_file, sep = ";", header = True, inferSchema = True)

Comprobemos que los datos se han cargado bien

In [ ]:
pmd.show(5)

~~~
+--------------------+--------------+
|            datetime|preciomd_eurMw|
+--------------------+--------------+
|2016-09-01 00:00:...|         37.76|
|2016-09-01 01:00:...|          36.0|
|2016-09-01 02:00:...|         35.26|
|2016-09-01 03:00:...|         35.26|
|2016-09-01 04:00:...|         35.26|
+--------------------+--------------+
only showing top 5 rows
~~~

#### P9. Imprime el esquema del DataFrame pmd

In [ ]:
# escribe aquí el código
pmd.printSchema()

#### P10. Ahora cruza los datos de consumo con los datos de precio del mercado diario que acabamos de cargar, trabajando directamente con los DataFrames

Pista: para cruzar DataFrames necesitarás usar un método que no hemos
visto en detalle durante la unidad: el método ***.join()***. 
> Este método permite cruzar dos DataFrames indicando los nombres de las columnas a comparar para hacer el cruce.

Para ello hay que tener **los 2 Dataframes cargados en el entorno**

- Fichero original o primero
- Fichero a cruzar o segundo

>****"nombre variable fichero primero".join("nombre variable fichero segundo", on = "columna comun o columna por la que se van a cruzar")***

In [ ]:
# escribe aquí el código
consumo_ext.join(pmd, on ="datetime")

#### P11. Repite la misma operación de la pregunta anterior, pero ahora registra el DataFrame pmd como una vista y utiliza una expresión SQL para cruzar los dos DataFrames. Verifica que el resultado es el mismo. Registra el resultado como una nueva vista.

Pista: Spark SQL permite las operaciones JOIN entre tablas como ocurre
en la mayoría de bases de datos relacionales.

In [ ]:
# escribe aquí el código


#### P12. Calcula la diferencia promedio para cada mes entre el precio pagado por el consumidor cada hora y el precio de la electricidad en el mercado diario. Trabaja directamente sobre el DataFrame creado en la pregunta P10.

Ojo, ten en cuenta que el precio del consumidor está en €/kWh y el
precio del mercado está en €/MWh.

In [ ]:
# escribe aquí el código


#### P13. Repite la misma operación de la pregunta anterior, pero ahora utilizando una expresión SQL sobre la vista resultado que registraste en la pregunta P11. Verifica que obtienes el mismo resultado.

In [ ]:
# escribe aquí el código
